# Session 3 — Behavior (clicks / RT), GSR, and pupil
**90 minutes**

### Goals
1. Build a clean multimodal timeline (stimulus onsets, clicks).
2. Compute **response time = first click − stimulus onset** (only if the click falls inside that stimulus window).
3. Summarize Tobii **GSR / SCR** without over-claiming causality.
4. Baseline-correct **pupil** within each stimulus; respect blinks as missingness.

No custom library beyond path helpers — calculations stay visible.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Readable plots for projection
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

SESSION_DIR = Path.cwd()
WORKSHOP_DIR = SESSION_DIR.parent if SESSION_DIR.name == "sessions" else Path("workshop")
sys.path.insert(0, str(WORKSHOP_DIR))
from analysis.paths import data_path, stimuli_path


## 1. Events on the teaching recording

In [ ]:
raw = pd.read_csv(data_path("food_decision_making", "Food_Decision_Making_Teaching_Sample.csv"))
raw["time_s"] = raw["Recording timestamp"].astype(float) / 1e3

events = raw.dropna(subset=["Event"]).copy()
event_counts = events["Event"].value_counts().rename_axis("Event").reset_index(name="n").head(8)
event_counts

In [ ]:
fig, ax = plt.subplots()
ax.barh(event_counts["Event"], event_counts["n"], color="#6b4c9a")
ax.set_xlabel("Count")
ax.set_title("Event types in teaching recording")
plt.tight_layout()
plt.show()

## 2. Stimulus intervals (TOIs) from start/end markers

We pair each `ImageStimulusStart` with the next `ImageStimulusEnd` **of the same name**.


In [ ]:
starts = events.loc[events["Event"] == "ImageStimulusStart", ["time_s", "Event value"]].rename(
    columns={"time_s": "onset_s", "Event value": "stimulus"}
)
ends = events.loc[events["Event"] == "ImageStimulusEnd", ["time_s", "Event value"]].rename(
    columns={"time_s": "offset_s", "Event value": "stimulus"}
)

intervals = []
for stim, sgrp in starts.groupby("stimulus"):
    egrp = ends.loc[ends["stimulus"] == stim].sort_values("offset_s")
    sgrp = sgrp.sort_values("onset_s")
    for (_, srow), (_, erow) in zip(sgrp.iterrows(), egrp.iterrows()):
        if erow["offset_s"] >= srow["onset_s"]:
            intervals.append({
                "stimulus": stim,
                "onset_s": srow["onset_s"],
                "offset_s": erow["offset_s"],
                "duration_s": erow["offset_s"] - srow["onset_s"],
            })
intervals = pd.DataFrame(intervals)
# Readable: food trials only
foods = ["cake", "pizza", "ice-cream", "cereal", "date"]
food_intervals = intervals.loc[intervals["stimulus"].isin(foods)].copy()
food_intervals.round(3)

## 3. Response times (stimulus onset → first mouse event in window)

Rational rule: only clicks with `onset ≤ click_time ≤ offset` count.  
If there is no click in the window, RT is missing (not zero).


In [ ]:
clicks = events.loc[events["Event"] == "MouseEvent", ["time_s", "Event value"]].copy()

rt_rows = []
for _, iv in food_intervals.iterrows():
    in_win = clicks[(clicks["time_s"] >= iv["onset_s"]) & (clicks["time_s"] <= iv["offset_s"])]
    if len(in_win) == 0:
        rt_rows.append({"stimulus": iv["stimulus"], "RT_s": np.nan, "n_clicks_in_window": 0})
        continue
    first = in_win.sort_values("time_s").iloc[0]
    rt_rows.append({
        "stimulus": iv["stimulus"],
        "RT_s": first["time_s"] - iv["onset_s"],
        "n_clicks_in_window": len(in_win),
        "first_click_value": first["Event value"],
    })
rt = pd.DataFrame(rt_rows)
rt["RT_s"] = rt["RT_s"].round(3)
rt

In [ ]:
plot_rt = rt.dropna(subset=["RT_s"])
fig, ax = plt.subplots()
if len(plot_rt):
    ax.bar(plot_rt["stimulus"], plot_rt["RT_s"], color="#8c2d4a")
    ax.set_ylabel("RT (s)")
    ax.set_title("First click after stimulus onset")
    plt.xticks(rotation=30, ha="right")
else:
    ax.text(0.5, 0.5, "No in-window clicks in this recording", ha="center")
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4. GSR metrics (aggregated Tobii table)

These are **interval-level** metrics from Tobii, not raw EDA samples.  
Good for teaching summaries; not a full SCR detection pipeline.


In [ ]:
gsr = pd.read_csv(data_path("tobii_gsr_demo", "Tobii_Pro_Lab_GSR_Demo_Project_Metrics.tsv"), sep="\t")
for c in ["Average_GSR", "Number_of_SCR", "Average_whole-fixation_pupil_diameter"]:
    if c in gsr.columns:
        gsr[c] = pd.to_numeric(gsr[c], errors="coerce")

# Participant-level means (keeps the table small)
part = (
    gsr.groupby("Participant", as_index=False)
    .agg(
        mean_GSR=("Average_GSR", "mean"),
        mean_SCR=("Number_of_SCR", "mean"),
        mean_pupil=("Average_whole-fixation_pupil_diameter", "mean"),
        n_intervals=("Average_GSR", "size"),
    )
)
part[["mean_GSR", "mean_SCR", "mean_pupil"]] = part[["mean_GSR", "mean_SCR", "mean_pupil"]].round(3)
part.head(8)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(part["mean_GSR"], part["mean_SCR"], s=60, alpha=0.8, color="#2a6f6f")
ax.set_xlabel("Mean Average_GSR")
ax.set_ylabel("Mean Number_of_SCR")
ax.set_title("Participants — GSR level vs SCR count")
plt.tight_layout()
plt.show()

### Optional: mouse-click metrics already computed by Tobii
Example column family: `Number_of_mouse_clicks.Snake` (AOI-specific).  
We show a **single** AOI summary, not the whole wide matrix.


In [ ]:
click_cols = [c for c in gsr.columns if c.startswith("Number_of_mouse_clicks.")]
# Prefer primary AOI columns without .1/.2 duplicates when possible
primary = [c for c in click_cols if c.count(".") == 1]
use_cols = primary[:4] if primary else click_cols[:4]
click_summary = gsr[use_cols].apply(pd.to_numeric, errors="coerce").mean().rename("mean_clicks")
click_summary = click_summary.reset_index().rename(columns={"index": "metric"})
click_summary["mean_clicks"] = click_summary["mean_clicks"].round(2)
click_summary

## 5. Pupil on the food teaching sample (baseline per stimulus)

**Baseline:** mean pupil in the first 0.5 s after stimulus onset (same stimulus rows only).  
**Corrected pupil** = pupil − baseline.  
This is a teaching baseline, not a full dilatory model.


In [ ]:
gaze = raw.loc[raw["Sensor"] == "Eye Tracker"].copy()
gaze["pupil"] = gaze[["Pupil diameter left", "Pupil diameter right"]].mean(axis=1, skipna=True)
gaze = gaze.dropna(subset=["pupil", "Presented Stimulus name", "time_s"])

pup_rows = []
for stim, grp in gaze.groupby("Presented Stimulus name"):
    grp = grp.sort_values("time_s")
    t0 = grp["time_s"].iloc[0]
    baseline = grp.loc[grp["time_s"] <= t0 + 0.5, "pupil"].mean()
    pup_rows.append({
        "stimulus": stim,
        "baseline_pupil_mm": baseline,
        "mean_pupil_mm": grp["pupil"].mean(),
        "mean_pupil_baseline_corrected": (grp["pupil"] - baseline).mean(),
        "n_samples": len(grp),
    })
pup_tbl = pd.DataFrame(pup_rows)
pup_tbl = pup_tbl.loc[pup_tbl["stimulus"].isin(foods + ["fixation", "instruction"])].copy()
pup_tbl = pup_tbl.round(3)
pup_tbl

In [ ]:
plot = pup_tbl.loc[pup_tbl["stimulus"].isin(foods)]
fig, ax = plt.subplots()
ax.bar(plot["stimulus"], plot["mean_pupil_baseline_corrected"], color="#3d5a80")
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Baseline-corrected pupil (mm)")
ax.set_title("Pupil by food stimulus (teaching sample)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Blinks (Pupil Labs) — keep the summary tiny

In [ ]:
blinks = pd.read_csv(data_path("pupil_labs_recording", "blinks.csv"))
blink_summary = pd.DataFrame({
    "n_blinks": [len(blinks)],
    "mean_duration_s": [round(blinks["duration"].mean(), 3)],
    "median_duration_s": [round(blinks["duration"].median(), 3)],
})
blink_summary

## Practice
1. Why is RT missing (`NaN`) more honest than putting `0` when no click occurs?
2. Name two confounds for GSR in a talking classroom demo.
3. Exit ticket: one sentence linking gaze (Session 2) to click RT (this session).
